In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import glob, os, re

TIME_CODER = xr.coders.CFDatetimeCoder(use_cftime=True)

MIRROR = "/global/cfs/cdirs/m3522/cmip6/CMIP6"
MODEL, MEMBER, GRID = "MPI-ESM1-2-HR", "r1i1p1f1", "gn"
ESGF_CACHE = os.path.expanduser("~/.esgf")          # where the hurs download landed

REPO = "/pscratch/sd/s/sarya4/github/HeatWaveAnalysis"
DATA = f"{REPO}/data"
os.makedirs(DATA, exist_ok=True)

CITIES = {
    "Phoenix":   (33.4484, 247.926),
    "Tucson":    (32.2226, 249.025),
    "Charlotte": (35.2271, 279.157),
    "Miami":     (25.7617, 279.808),
}

WARM_MONTHS = [5, 6, 7, 8, 9]
BASELINE = (1996, 2026)
FUTURE   = (2027, 2057)

In [4]:
# hurs comes from three places, because the NERSC mirror's historical
# record has a hole in it:
# 1996-1999  -> mirror  (CMIP/MPI-M)
# 2000-2014  -> intake-esgf download in ~/.esgf
# 2015-2057  -> mirror  (ScenarioMIP/DKRZ)

def in_window(path, y0=1996, y1=2057):
    """Keep only files overlapping the study window. The mirror also holds
    stray chunks (1910-14, 1955-59) we don't want pulled in."""
    m = re.search(r"_(\d{8})-(\d{8})\.nc$", path)
    if not m:
        return False
    a, b = int(m.group(1)[:4]), int(m.group(2)[:4])
    return not (b < y0 or a > y1)

def hurs_files():
    mirror_hist = glob.glob(f"{MIRROR}/CMIP/MPI-M/{MODEL}/historical/{MEMBER}/day/hurs/{GRID}/*/*.nc")
    cached_hist = glob.glob(f"{ESGF_CACHE}/**/day/hurs/**/*.nc", recursive=True)
    mirror_ssp  = glob.glob(f"{MIRROR}/ScenarioMIP/DKRZ/{MODEL}/ssp370/{MEMBER}/day/hurs/{GRID}/*/*.nc")
    files = [f for f in set(mirror_hist + cached_hist + mirror_ssp) if in_window(f)]
    files.sort(key=lambda p: re.search(r"_(\d{8})-\d{8}\.nc$", p).group(1))   # by start date
    for f in files:
        print("  ", os.path.basename(f))
    return files

HURS_FILES = hurs_files()
print(f"\n{len(HURS_FILES)} files in window")

   hurs_day_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_19950101-19991231.nc
   hurs_day_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_20000101-20041231.nc
   hurs_day_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_20050101-20091231.nc
   hurs_day_MPI-ESM1-2-HR_historical_r1i1p1f1_gn_20100101-20141231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20150101-20191231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20200101-20241231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20250101-20291231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20300101-20341231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20350101-20391231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20400101-20441231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20450101-20491231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20500101-20541231.nc
   hurs_day_MPI-ESM1-2-HR_ssp370_r1i1p1f1_gn_20550101-20591231.nc

13 files in window


In [5]:
# nearest-gridcell extraction.
# hurs is already relative humidity in percent.

ds = xr.open_mfdataset(HURS_FILES, combine="by_coords", decode_times=TIME_CODER,
                       data_vars="minimal", coords="minimal", compat="override")
da = ds["hurs"].sel(time=slice("1996", "2057"))

series = []
for city, (lat, lon) in CITIES.items():
    pt = da.sel(lat=lat, lon=lon, method="nearest")
    print(f"{city:9s} -> grid lat {float(pt.lat):6.2f}, lon {float(pt.lon):7.2f}")
    series.append(pt.drop_vars(["lat", "lon"]))
hurs = xr.concat(series, dim=pd.Index(list(CITIES), name="city")).load()

# the same continuity check we ran on temperature -- this is the one that
# proves the mirror + download + ssp370 splice has no hole in it
tv = hurs.time.values
steps = np.array([(tv[i+1] - tv[i]).days for i in range(len(tv) - 1)])
print("\nday-steps (want [1] only):", np.unique(steps))
print("time:", str(tv[0])[:10], "->", str(tv[-1])[:10], "| days:", len(tv))
print("humidity range: %.1f%% to %.1f%%" % (float(hurs.min()), float(hurs.max())))

Phoenix   -> grid lat  33.19, lon  247.50
Tucson    -> grid lat  32.26, lon  249.38
Charlotte -> grid lat  35.06, lon  279.38
Miami     -> grid lat  25.71, lon  279.38

day-steps (want [1] only): [1]
time: 1996-01-01 -> 2057-12-31 | days: 22646
humidity range: 5.0% to 103.7%


In [6]:
hurs_ds = xr.Dataset({"hurs": hurs})
hurs_ds.attrs.update(model=MODEL, member=MEMBER, units="%",
                     experiments="historical 1996-2014 (mirror + ESGF download) + ssp370 2015-2057")
hurs_ds.to_netcdf(f"{DATA}/city_daily_hurs_1996_2057.nc")

dfh = hurs_ds.to_dataframe().reset_index()
dfh["time"] = dfh["time"].astype(str)
dfh.to_csv(f"{DATA}/city_daily_hurs_1996_2057.csv", index=False)
print("saved hurs to", DATA)

saved hurs to /pscratch/sd/s/sarya4/github/HeatWaveAnalysis/data
